# 🧩 Prerequisites

In [ ]:
# @title 📀 Mount Google Drive
from google.colab import drive
from pathlib import Path
drive.mount("/content/drive")
%cd /content/drive/MyDrive/Colab/thesis/rrs
CWD = Path.cwd()

In [ ]:
%%capture
!pip install uv
!uv pip install evaluate rouge_score bert-score
!uv pip install ctranslate2

# 📊 Dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset("adasatwork-thesis/rrs", data_files={
    "train": "train.jsonl",
    "eval" : "dev.jsonl",
    "test" : "test.jsonl"
})

print(dataset)

# 🚀 Train Model

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "GanjinZero/biobart-v2-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [ ]:
def preprocess(example):
    model_inputs = tokenizer(
        example["findings"],
        max_length=512,
        padding=True,
        truncation=True
    )

    labels = tokenizer(
        example["impression"],
        max_length=256,
        padding=True,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs


tokenized_dataset = dataset.map(
    preprocess,
    batched=True,
    remove_columns=dataset["train"].column_names
)

In [ ]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="/content/outputs",
    eval_strategy="steps",
    save_strategy="no",
    eval_steps=100,
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=6,
    predict_with_generate=True,
    generation_max_length=256,
    generation_num_beams=4,
    bf16=True, tf32=True,
    logging_steps=20,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    seed = 3407,
    report_to = "none",
)

In [ ]:
from transformers import Seq2SeqTrainer, DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model,
    training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["eval"],
    data_collator=data_collator,
    processing_class=tokenizer
)

In [ ]:
trainer.train()

In [ ]:
trainer.evaluate()

In [ ]:
trainer.save_model("models/finetuned_biobart")
tokenizer.save_pretrained("models/finetuned_biobart")

# 🔥 Inference

In [ ]:
# !ct2-transformers-converter --model models/finetuned_biobart --output_dir models/finetuned_biobart-ct2 --quantization int8_float16

In [ ]:
# @title BioBART Inference Using Core HuggingFace Transformer
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

class BioBART:
    def __init__(self, model_name):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name, device_map="auto")
        self.model.eval()

    def __call__(self, text):
        return self.generate(text)

    def generate(self, text):
        inputs = self.tokenizer(
            text,
            max_length=512,
            padding=True,
            truncation=True,
            return_tensors="pt",
        ).to("cuda")

        with torch.inference_mode():
            with torch.amp.autocast("cuda"):
                outputs = self.model.generate(
                    **inputs,
                    max_length=256,
                    num_beams=4,
                    length_penalty=0.8,
                    repetition_penalty=1.5,
                )

        return self.tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
# @title BioBART Inference Using CTranslate2
import torch
import ctranslate2
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

class BioBART:
    def __init__(self, model_name):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.translator = ctranslate2.Translator(
            model_name + "-ct2",
            device="cuda",
            compute_type="float16"
        )

    def __call__(self, batch):
        return self.generate(batch)

    def generate(self, batch):
        texts = batch["source"]
        inputs = self.tokenizer(
            texts,
            max_length=512,
            truncation=True
        )
        batch_tokens = [
            self.tokenizer.convert_ids_to_tokens(ids)
            for ids in inputs["input_ids"]
        ]

        results = self.translator.translate_batch(
            batch_tokens,
            beam_size=4,
            patience=1.5,
            max_decoding_length=256,
            min_decoding_length=10,
            length_penalty=0.8,
            repetition_penalty=1.5,
            max_batch_size=64
        )

        outputs = self.tokenizer.batch_decode(
            [self.tokenizer.convert_tokens_to_ids(r.hypotheses[0]) for r in results],
            skip_special_tokens=True
        )

        return {"biobart_gen": outputs}